# 4 - Guide d'intégration des modèles de différents packages

Ce notebook montre comment utiliser des modèles issus de `scikit-learn`, `xgboost`, `tslearn`, `sktime` et `darts` dans un même cadre de prévision en pseudo temps réel, avec les découpages de `tsforecast.crossvals`, les pipelines de `scikit-learn` et `tsforecast.xy`, et les adapters de `tsforecast.adapters`.

Chaque modèle est présenté selon le même gabarit : principe, série temporelle, panel, `Pipeline` / `XYPipeline`, `cross_val_score`, `GridSearchCV`, limites.

Les données sont synthétiques et déterministes. Dépendances : `uv sync --group notebooks`.

## Importation des modules

In [1]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Importation des modules
# Modules de base
import sys
import warnings
import zlib

# Manipulation de données
import numpy as np
import pandas as pd

# Ajout du chemin
sys.path.append('..')

# Sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

#  Eléments du package
from tsforecast.crossvals import PanelOutOfSampleSplit, TSOutOfSampleSplit
from tsforecast.xy import DropNaNTarget, XYPipeline

## 1. Création de données synthétiques

### 1.1 Génération de données de panel

In [2]:
# Fonction de génération des données de panel
def generate_panel_data(entities=['A', 'B', 'C'], start_date='2015-01-01', periods=120,
                        freq='MS', heterogeneous_effects=True, common_trend=True,
                        entity_specific_seasonality=True, cross_sectional_correlation=0.3,
                        indicator_lead=3, missing_data_prob=0.0, seed=42):
    """
    Generate synthetic panel data with various characteristics.

    Args:
        entities: List of entity identifiers
        start_date: Start date for the panel
        periods: Number of time periods per entity
        freq: Frequency of observations
        heterogeneous_effects: Whether entities have different baseline levels
        common_trend: Whether to include a common trend across entities
        entity_specific_seasonality: Whether seasonality patterns differ by entity
        cross_sectional_correlation: Correlation between entity shocks
        indicator_lead: Number of periods by which the indicator leads the value
        missing_data_prob: Probability of missing observations
        seed: Seed of the random generator

    Returns:
        pd.DataFrame: Panel data with MultiIndex (entity, date)
    """
    # Initialisation du générateur aléatoire
    rng = np.random.default_rng(seed)

    # Création de l'index temporel
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)

    # Création du MultiIndex (entity, date)
    index = pd.MultiIndex.from_product([entities, dates], names=['entity', 'date'])

    # Génération des effets fixes par entité (hétérogénéité)
    if heterogeneous_effects:
        entity_effects = {entity: rng.normal(0, 2) for entity in entities}
    else:
        entity_effects = {entity: 0 for entity in entities}

    # Tendance commune
    if common_trend:
        common_trend_values = 0.02 * np.arange(periods)
    else:
        common_trend_values = np.zeros(periods)

    # Chocs communs aux entités
    common_shocks = rng.normal(0, 1, periods)

    # Construction des séries pour chaque entité
    values, indicators = [], []

    # Parcours des entités
    for entity in entities:
        # Code de l'entité, stable d'une exécution à l'autre
        code = zlib.crc32(str(entity).encode())

        # Saisonnalité spécifique à l'entité
        if entity_specific_seasonality:
            # Période et amplitude différentes selon l'entité
            seasonal_period = 20 + code % 40  # Entre 20 et 60
            seasonal_amplitude = 0.5 + (code % 100) / 200  # Entre 0.5 et 1.0
        else:
            seasonal_period = 30
            seasonal_amplitude = 0.5

        seasonal_values = seasonal_amplitude * np.sin(2 * np.pi * np.arange(periods) / seasonal_period)

        # Processus autorégressif spécifique à l'entité
        ar_coef = 0.5 + (code % 40) / 100  # Entre 0.5 et 0.9
        ar_process = np.zeros(periods)
        ar_process[0] = rng.normal(0, 0.5)
        for t in range(1, periods):
            ar_process[t] = ar_coef * ar_process[t-1] + rng.normal(0, 0.3)

        # Indicateur avancé : cycle bruité, prolongé en amont de la première date
        steps = np.arange(periods + indicator_lead)
        indicator = np.sin(2 * np.pi * steps / 24 + code % 7) + rng.normal(0, 0.3, len(steps))

        # Chocs avec corrélation croisée
        correlated_shocks = 0.5 * (
            np.sqrt(cross_sectional_correlation) * common_shocks +
            np.sqrt(1 - cross_sectional_correlation) * rng.normal(0, 1, periods)
        )

        # Combinaison des composantes : la valeur suit l'indicateur avec retard
        entity_values = (
            entity_effects[entity] +
            common_trend_values +
            seasonal_values +
            ar_process +
            2.0 * indicator[:periods] +
            correlated_shocks
        )

        # Ajout de données manquantes
        if missing_data_prob > 0:
            missing_mask = rng.random(periods) < missing_data_prob
            entity_values[missing_mask] = np.nan

        values.extend(entity_values)
        indicators.extend(indicator[indicator_lead:])

    # Création du DataFrame final
    panel_data = pd.DataFrame({'value': values, 'indicator': indicators}, index=index)

    # Ajout de variables calendaires
    panel_data['trend'] = np.tile(np.arange(periods), len(entities))
    panel_data['month'] = panel_data.index.get_level_values('date').month

    return panel_data

In [3]:
# Génération de différents types de données de panel
print("📊 Génération de données de panel ...")

# Panel 1: Données équilibrées avec effets hétérogènes
entities_small = ['FR', 'DE', 'IT', 'ES']
df_panel = generate_panel_data(
    entities=entities_small,
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    common_trend=True,
    entity_specific_seasonality=True,
    cross_sectional_correlation=0.4
)

print(f"✅ Génération de panels terminée:")
print(f"Caractéristiques des données générées : {df_panel.shape[0]} observations, {len(entities_small)} entités")

# Affichage des premières observations de chaque panel
print(f"\n📋 Aperçu des données:")
df_panel.head(8)

📊 Génération de données de panel ...
✅ Génération de panels terminée:
Caractéristiques des données générées : 480 observations, 4 entités

📋 Aperçu des données:


value  indicator  trend  month
entity date                                         
FR     2015-01-01  3.187705   0.899784      0      1
       2015-02-01  1.607159   0.965775      1      2
       2015-03-01  2.511569  -0.518093      2      3
       2015-04-01  2.943949  -0.526622      3      4
       2015-05-01  3.552726  -0.534844      4      5
       2015-06-01 -0.029454  -0.296433      5      6
       2015-07-01  0.541955  -1.233288      6      7
       2015-08-01  0.134507  -1.069131      7      8

### 1.2 Génération de données hiérarchiques

Pour illustrer la réconciliation hiérarchique, on génère des séries `value_b` et `value_c` 
telles que `value_a = value_b + value_c`. Cette contrainte d'agrégation est la base 
des hiérarchies cross-sectionnelles : les prévisions indépendantes de chaque composante 
ne respectent pas nécessairement la contrainte, et la réconciliation permet de les rendre cohérentes.

In [4]:
def generate_hierarchical_data(
    entities=['X', 'Y', 'Z'],
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    seed=42,
):
    """Generate hierarchical panel data where value_a = value_b + value_c and value_c = value_d + value_e.

    Generates two bottom-level series (value_b, value_c) per entity, with
    distinct trend, seasonality and noise patterns. The top-level series
    (value_a) is their exact sum, forming a cross-sectional aggregation
    constraint. Data is indexed by (entity, date).

    Args:
        entities: List of entity identifiers.
        start_date: Start date for the time series.
        periods: Number of time periods per entity.
        freq: Frequency of observations.
        heterogeneous_effects: Whether entities have different baseline levels
            and trend slopes for value_b and value_c.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (entity, date) and columns value_a, value_b, value_c, value_d, value_e.
    """
    # Initialisation du seed
    np.random.seed(seed)
    # Initialisation des dates
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    t = np.arange(periods)

    # Liste des observations hiérarchiques
    hierarchical_records = []

    # Parcours des entités
    for entity in entities:
        # Effets fixes par entité (niveaux et pentes)
        if heterogeneous_effects:
            base_b = np.random.uniform(40, 70)
            base_d = np.random.uniform(20, 45)
            base_e = np.random.uniform(10, 25)
            slope_b = np.random.uniform(0.05, 0.25)
            slope_d = np.random.uniform(-0.10, 0.05)
            slope_e = np.random.uniform(-0.20, 0.15)
        else:
            base_b, base_d, base_e = 50, 30, 10
            slope_b, slope_d, slope_e = 0.15, -0.05, -0.15

        # Composante B : tendance + saisonnalité annuelle + bruit
        value_b = (
            base_b
            + slope_b * t
            + 5 * np.sin(2 * np.pi * t / 12)
            + np.random.normal(0, 1.5, periods)
        )

        # Composante D : tendance + saisonnalité semestrielle + bruit
        value_d = (
            base_d
            + slope_d * t
            + 3 * np.sin(2 * np.pi * t / 6)
            + np.random.normal(0, 1.0, periods)
        )

        # Composante E : tendance + saisonnalité semestrielle + bruit
        value_e = (
            base_e
            + slope_e * t
            + 3 * np.sin(2 * np.pi * t / 6)
            + np.random.normal(0, 1.0, periods)
        )

        # Agrégat : contrainte d'agrégation exacte
        value_c = value_d + value_e
        value_a = value_b + value_c

        # Accumulation des lignes pour le format wide
        for d, va, vb, vc, vd, ve in zip(dates, value_a, value_b, value_c, value_d, value_e):
            hierarchical_records.append({
                'entity': entity, 'date': d,
                'value_a': va, 'value_b': vb, 'value_c': vc, 'value_d': vd, 'value_e': ve,
            })

    # Format wide : MultiIndex (entity, date)
    df_hierarchical = pd.DataFrame(hierarchical_records).set_index(['entity', 'date'])

    return df_hierarchical


# Génération des données hiérarchiques
print('📊 Génération de données hiérarchiques ...')
df_hier = generate_hierarchical_data(
    entities=['FR', 'DE', 'IT', 'ES'],
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
)

n_entities = df_hier.index.get_level_values('entity').nunique()
constraint_ok = (
    df_hier
    .assign(check=lambda d: np.isclose(d['value_a'], d['value_b'] + d['value_c']))
    ['check']
    .all()
)

print('✅ Données hiérarchiques générées:')
print(f'   Format wide : {df_hier.shape} (MultiIndex entity/date, {n_entities} entités)')
print(f'   Vérification : value_a == value_b + value_c → {constraint_ok}')

print('\n📋 Aperçu format wide :')
print(df_hier.head(8))

📊 Génération de données hiérarchiques ...
✅ Données hiérarchiques générées:
   Format wide : (480, 5) (MultiIndex entity/date, 4 entités)
   Vérification : value_a == value_b + value_c → True

📋 Aperçu format wide :
                      value_a    value_b    value_c    value_d    value_e
entity date                                                              
FR     2015-01-01  116.887308  53.605023  63.282285  42.777321  20.504964
       2015-02-01  123.559380  55.057087  68.502293  45.723039  22.779254
       2015-03-01  126.566609  55.201582  71.365027  46.312391  25.052636
       2015-04-01  121.542514  57.559239  63.983275  43.034590  20.948685
       2015-05-01  111.402201  55.550131  55.852070  39.312729  16.539341
       2015-06-01  113.314311  53.886267  59.428044  40.855358  18.572685
       2015-07-01  117.093162  52.617537  64.475625  42.245971  22.229654
       2015-08-01  116.950388  47.054405  69.895983  46.303346  23.592637


## 2. Création d'un cadre de prévision à partir des éléments développés dans `tsforecast`

La construction suit le tutoriel `docs/tutorials/pseudo_real_time_forecast.md` :

- une ligne est indexée par son **origine** `t`, date à laquelle la prévision est faite ;
- la cible de la ligne `t` est `y(t + h)` : elle est décalée de `-h`, **entité par entité** ;
- les covariables de la ligne `t` sont celles **connues à l'origine** : la valeur de la cible n'y entre que décalée de son délai de publication `d` ;
- le découpage laisse `gap = h + d - 1` lignes entre l'entraînement et le test.

Les modèles apprennent `y` à partir de `X` ligne à ligne : plus rien n'est à décaler ensuite, ni dans les pipelines, ni dans les adapters.

### 2.1 Jeu de données supervisé

L'indicateur avancé `indicator` est supposé publié sans délai ; la valeur `value` l'est avec un mois de retard.

In [5]:
# Horizon de prévision
h = 2
# Délai de publication de la cible
d = 1
# Écart entre l'entraînement et le test
gap = h + d - 1

# Décalages entité par entité : un shift sur tout le panel mélangerait les entités
by_entity = df_panel.groupby(level='entity')
supervised = pd.DataFrame({
    'value_published': by_entity['value'].shift(d),           # dernière valeur publiée en t
    'value_published_lag1': by_entity['value'].shift(d + 1),
    'indicator': df_panel['indicator'],
    'indicator_lag1': by_entity['indicator'].shift(1),
    'trend': df_panel['trend'],
    'month': df_panel['month'],
    'target': by_entity['value'].shift(-h),                   # valeur à prévoir, y(t + h)
})
features = [column for column in supervised.columns if column != 'target']

# Origines dont toutes les covariables sont disponibles
supervised = supervised.dropna(subset=features)
# Origines dont la cible est observée : jeu d'apprentissage et d'évaluation
labelled = supervised.dropna(subset=['target'])
X, y = labelled[features], labelled['target']

# Série temporelle : une seule entité, index de dates
X_ts, y_ts = X.loc['FR'], y.loc['FR']

print(f'Panel            : {X.shape}, origines {X.index.levels[1].min():%Y-%m} → {X.index.get_level_values("date").max():%Y-%m}')
print(f'Série temporelle : {X_ts.shape}')
X_ts.assign(target=y_ts).head()

Panel            : (464, 6), origines 2015-01 → 2024-10


Série temporelle : (116, 6)


,value_published,value_published_lag1,indicator,indicator_lag1,trend,month,target
date,,,,,,,
2015-03-01,1.607159,3.187705,-0.518093,0.965775,2,3,3.552726
2015-04-01,2.511569,1.607159,-0.526622,-0.518093,3,4,-0.029454
2015-05-01,2.943949,2.511569,-0.534844,-0.526622,4,5,0.541955
2015-06-01,3.552726,2.943949,-0.296433,-0.534844,5,6,0.134507
2015-07-01,-0.029454,3.552726,-1.233288,-0.296433,6,7,1.492616


### 2.2 Découpages et contrôle de l'absence de fuite

`TSOutOfSampleSplit` découpe une série, `PanelOutOfSampleSplit` un panel (mêmes dates de test pour toutes les entités). Le dernier pli sert d'illustration pour un ajustement unique ; des découpages de validation, appliqués à son seul entraînement, servent à `GridSearchCV`.

In [6]:
# Découpages d'évaluation
cv_ts = TSOutOfSampleSplit(n_splits=4, test_size=6, gap=gap)
cv_panel = PanelOutOfSampleSplit(n_splits=4, test_size=6, gap=gap)
# Découpages de validation des hyperparamètres
cv_val_ts = TSOutOfSampleSplit(n_splits=3, test_size=6, gap=gap)
cv_val_panel = PanelOutOfSampleSplit(n_splits=3, test_size=6, gap=gap)

# Dernier pli : un jeu d'entraînement et un jeu de test
train, test = list(cv_ts.split(X_ts))[-1]
X_ts_train, y_ts_train = X_ts.iloc[train], y_ts.iloc[train]
X_ts_test, y_ts_test = X_ts.iloc[test], y_ts.iloc[test]

train, test = list(cv_panel.split(X))[-1]
X_train, y_train = X.iloc[train], y.iloc[train]
X_test, y_test = X.iloc[test], y.iloc[test]

# Contrôle : le modèle apprend-il une cible inconnue à la première origine de test ?
def leaks(splitter, X):
    """Tell, for each fold, whether a training target is unpublished at the first test origin."""
    origins = X.index.get_level_values(-1).to_period('M')
    return [origins[train].max() + h > origins[test].min() - d for train, test in splitter.split(X)]

for value in range(gap + 1):
    print(f'gap={value} : fuite par pli {leaks(PanelOutOfSampleSplit(n_splits=4, test_size=6, gap=value), X)}')

print(f'\nSérie : entraînement → {X_ts_train.index.max():%Y-%m}, test {X_ts_test.index.min():%Y-%m} → {X_ts_test.index.max():%Y-%m}')
print(f'Panel : {len(X_train)} lignes d\'entraînement, {len(X_test)} lignes de test')

gap=0 : fuite par pli [True, True, True, True]
gap=1 : fuite par pli [True, True, True, True]
gap=2 : fuite par pli [False, False, False, False]

Série : entraînement → 2024-02, test 2024-05 → 2024-10
Panel : 432 lignes d'entraînement, 24 lignes de test


### 2.3 Prévision en conditions réelles

À la dernière origine `T`, on veut prévoir `y(T + h)`. Seules les cibles des origines `s ≤ T - h - d` sont publiées : les cibles suivantes sont masquées, **sans retirer les lignes**. Dans une `XYPipeline`, l'étape `DropNaNTarget` les écarte de l'apprentissage juste avant le modèle, et jamais de la prédiction.

In [7]:
# Dernière origine disponible
T = supervised.index.get_level_values('date').max()
# Cibles publiées à l'origine T
published = supervised.index.get_level_values('date') <= T - pd.DateOffset(months=h + d)

X_live = supervised[features]
y_live = supervised['target'].where(published)
X_live_ts, y_live_ts = X_live.loc['FR'], y_live.loc['FR']

# Lignes à prévoir : origine T, pour chaque entité et pour la série
X_now = X_live[X_live.index.get_level_values('date') == T]
X_now_ts = X_live_ts.loc[[T]]

print(f'Origine T = {T:%Y-%m}, période prévue : {T + pd.DateOffset(months=h):%Y-%m}')
y_live_ts.tail(5).to_frame()

Origine T = 2024-12, période prévue : 2025-02


,target
date,
2024-08-01,4.678249
2024-09-01,4.683252
2024-10-01,NaN
2024-11-01,NaN
2024-12-01,NaN


## 3. Intégration des modèles de différents packages dans un cadre unifié, compatible avec les éléments développés dans `tsforecast`

Quel que soit le package d'origine, un modèle s'utilise comme un régresseur `scikit-learn` : `fit(X, y)` apprend `y` à partir de `X`, `predict(X)` renvoie un `numpy.ndarray` dans l'ordre des lignes de `X`. Les adapters de `tsforecast.adapters` ne modifient jamais l'objet qu'on leur confie : `fit` en ajuste une copie.

In [8]:
# Mise en regard des observations et des prévisions d'un jeu de test
def compare(y_true, y_pred):
    """Align the observed and predicted values, and print the mean absolute error."""
    print(f'MAE sur le test : {mean_absolute_error(y_true, y_pred):.3f}')
    return pd.DataFrame({'observé': y_true, 'prévu': y_pred}, index=y_true.index).round(3)

# Affichage du message d'une erreur attendue
def show_error(call):
    """Run a call expected to fail and print its error message."""
    try:
        call()
    except ValueError as error:
        print(f'{type(error).__name__} : {error}')

### 3.1. Intégration des modèles `sklearn`

#### 3.1.1. Principe

L'intégration des estimateurs de `sklearn` est native, `X` et `y` étant déjà alignés. `GridSearchCV`, `cross_val_score`, `Pipeline`, etc. s'utilisent avec leur syntaxe d'origine ; les découpages de `tsforecast.crossvals` se passent en `cv`.

#### 3.1.2. Série temporelle

In [9]:
estimator = LinearRegression().fit(X_ts_train, y_ts_train)
compare(y_ts_test, estimator.predict(X_ts_test))

MAE sur le test : 0.514

,observé,prévu
date,,
2024-05-01,4.013,3.157
2024-06-01,3.747,3.987
2024-07-01,3.887,4.321
2024-08-01,4.678,4.801
2024-09-01,4.683,5.215
2024-10-01,3.267,4.167


#### 3.1.3. Panel

Un estimateur `sklearn` ne lit pas l'index : il ajuste **un seul modèle** sur les lignes empilées de toutes les entités.

In [10]:
estimator = LinearRegression().fit(X_train, y_train)
compare(y_test, estimator.predict(X_test)).unstack('entity').head()

MAE sur le test : 0.639


observé                       prévu                     
entity          DE     ES     FR     IT     DE     ES     FR     IT
date                                                               
2024-05-01  -0.510  6.877  4.013  6.360 -0.239  5.851  2.966  5.709
2024-06-01  -2.606  4.770  3.747  6.204 -1.761  5.400  3.914  5.585
2024-07-01  -1.373  4.416  3.887  4.458 -1.616  5.315  4.210  5.232
2024-08-01  -1.475  3.800  4.678  4.511 -0.760  5.112  4.708  5.371
2024-09-01  -0.069  3.071  4.683  5.268 -0.629  4.078  4.803  5.768

#### 3.1.4. `Pipeline` et `XYPipeline`

In [11]:
# Pipeline sklearn
pipeline = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])
pipeline.fit(X_train, y_train)
print(f'Pipeline, MAE : {mean_absolute_error(y_test, pipeline.predict(X_test)):.3f}')

# XYPipeline : les origines sans cible publiée sont écartées de l'apprentissage seulement
pipeline = XYPipeline([('drop', DropNaNTarget()), ('scaler', StandardScaler()), ('model', Ridge())])
pipeline.fit(X_live, y_live)
pd.Series(pipeline.predict(X_now), index=X_now.index, name=f'prévision faite en {T:%Y-%m}').round(3)

Pipeline, MAE : 0.638


entity  date      
FR      2024-12-01    5.143
DE      2024-12-01    0.198
IT      2024-12-01    3.891
ES      2024-12-01    2.270
Name: prévision faite en 2024-12, dtype: float64

#### 3.1.5. `cross_val_score`

In [12]:
for name, data, target, cv in [('série', X_ts, y_ts, cv_ts), ('panel', X, y, cv_panel)]:
    scores = cross_val_score(Ridge(), data, target, cv=cv, scoring='neg_mean_absolute_error')
    print(f'{name} : MAE par pli {(-scores).round(3)}, moyenne {-scores.mean():.3f}')

série : MAE par pli [0.621 1.218 0.689 0.501], moyenne 0.757


panel : MAE par pli [0.77  1.364 1.156 0.64 ], moyenne 0.982


#### 3.1.6. `GridSearchCV`

In [13]:
search = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('model', Ridge())]),
    param_grid={'model__alpha': [0.1, 1.0, 10.0, 100.0]},
    cv=cv_val_panel,
    scoring='neg_mean_absolute_error',
).fit(X_train, y_train)

print(f'Meilleurs paramètres : {search.best_params_}, MAE de validation : {-search.best_score_:.3f}')
print(f'MAE sur le test      : {mean_absolute_error(y_test, search.predict(X_test)):.3f}')

Meilleurs paramètres : {'model__alpha': 100.0}, MAE de validation : 1.079
MAE sur le test      : 0.600


#### 3.1.7. Limites et pièges

- Sur un panel, le modèle mutualisé ignore l'identité des entités : ajouter des indicatrices d'entité dans `X`, ou recourir à un modèle local par entité (section 3.4.3).
- La plupart des estimateurs refusent une cible manquante : placer `DropNaNTarget` dans une `XYPipeline`, plutôt que de retirer des lignes de `X` avant le découpage.
- `cross_val_score` utilise par défaut le R², instable sur des tests de quelques lignes : préciser `scoring`.
- Un `KFold` ordinaire apprendrait sur le futur : toujours passer un découpage temporel avec le bon `gap`.

### 3.2. Intégration des modèles `xgboost`

#### 3.2.1. Principe

L'API de `xgboost` suit celle de `sklearn` : `XGBRegressor` s'utilise sans adapter, avec les mêmes outils.

#### 3.2.2. Série temporelle

In [14]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=0)
compare(y_ts_test, xgb.fit(X_ts_train, y_ts_train).predict(X_ts_test))

MAE sur le test : 0.518


,observé,prévu
date,,
2024-05-01,4.013,2.390
2024-06-01,3.747,3.807
2024-07-01,3.887,4.286
2024-08-01,4.678,4.570
2024-09-01,4.683,4.390
2024-10-01,3.267,3.891


#### 3.2.3. Panel

In [15]:
compare(y_test, xgb.fit(X_train, y_train).predict(X_test)).unstack('entity').head()

MAE sur le test : 0.744


observé                       prévu                     
entity          DE     ES     FR     IT     DE     ES     FR     IT
date                                                               
2024-05-01  -0.510  6.877  4.013  6.360  0.037  5.330  3.878  5.192
2024-06-01  -2.606  4.770  3.747  6.204 -1.707  4.787  4.789  5.523
2024-07-01  -1.373  4.416  3.887  4.458 -1.704  3.492  5.161  4.711
2024-08-01  -1.475  3.800  4.678  4.511 -0.625  3.437  5.372  4.614
2024-09-01  -0.069  3.071  4.683  5.268 -0.734  3.150  5.361  4.983

#### 3.2.4. `Pipeline` et `XYPipeline`

In [16]:
pipeline = Pipeline([('scaler', StandardScaler()), ('model', xgb)])
pipeline.fit(X_train, y_train)
print(f'Pipeline, MAE : {mean_absolute_error(y_test, pipeline.predict(X_test)):.3f}')

pipeline = XYPipeline([('drop', DropNaNTarget()), ('model', xgb)])
pipeline.fit(X_live, y_live)
pd.Series(pipeline.predict(X_now), index=X_now.index, name=f'prévision faite en {T:%Y-%m}').round(3)

Pipeline, MAE : 0.744


entity  date      
FR      2024-12-01    4.657
DE      2024-12-01    0.007
IT      2024-12-01    3.117
ES      2024-12-01    2.250
Name: prévision faite en 2024-12, dtype: float32

#### 3.2.5. `cross_val_score`

In [17]:
for name, data, target, cv in [('série', X_ts, y_ts, cv_ts), ('panel', X, y, cv_panel)]:
    scores = cross_val_score(xgb, data, target, cv=cv, scoring='neg_mean_absolute_error')
    print(f'{name} : MAE par pli {(-scores).round(3)}, moyenne {-scores.mean():.3f}')

série : MAE par pli [1.263 1.836 0.88  0.518], moyenne 1.124


panel : MAE par pli [0.827 1.652 1.52  0.744], moyenne 1.186


#### 3.2.6. `GridSearchCV`

In [18]:
search = GridSearchCV(
    xgb,
    param_grid={'max_depth': [2, 3], 'learning_rate': [0.05, 0.1]},
    cv=cv_val_panel,
    scoring='neg_mean_absolute_error',
).fit(X_train, y_train)

print(f'Meilleurs paramètres : {search.best_params_}, MAE de validation : {-search.best_score_:.3f}')
print(f'MAE sur le test      : {mean_absolute_error(y_test, search.predict(X_test)):.3f}')

Meilleurs paramètres : {'learning_rate': 0.05, 'max_depth': 2}, MAE de validation : 1.116
MAE sur le test      : 0.757


#### 3.2.7. Limites et pièges

- Des arbres n'extrapolent pas : une cible à tendance sort de l'intervalle appris. Modéliser des variations plutôt que des niveaux, ou combiner avec un modèle linéaire.
- La mise à l'échelle est sans effet utile sur des arbres.
- `XGBRegressor` utilise déjà plusieurs cœurs : `n_jobs` dans `GridSearchCV` en plus sursollicite la machine.

### 3.3. Intégration des modèles `tslearn`

#### 3.3.1. Principe

Les estimateurs de `tslearn` suivent l'API de `sklearn`, mais **chaque ligne de `X` est lue comme une série** : un tableau `(n_lignes, n_colonnes)` désigne `n_lignes` séries univariées de longueur `n_colonnes`. Les noyaux (GAK) et distances (DTW) comparent des formes, ce qui n'a de sens que si les colonnes sont les valeurs successives d'une même grandeur. On construit donc, pour chaque origine, la fenêtre des douze dernières valeurs publiées, de la plus ancienne à la plus récente.

In [19]:
# Filtrage de l'avertissement de dépendance optionnelle à l'import
warnings.filterwarnings('ignore', message='h5py not installed')

from tslearn.neighbors import KNeighborsTimeSeriesRegressor
from tslearn.svm import TimeSeriesSVR

# Longueur de la fenêtre
window = 12

# Fenêtre des valeurs publiées à chaque origine, entité par entité
windows = pd.DataFrame({
    f'published_lag{lag}': by_entity['value'].shift(d + lag)
    for lag in reversed(range(window))
}).dropna()

# Cible alignée sur les origines, masquée quand elle n'est pas publiée en T
X_win_live = windows.loc[windows.index.intersection(X_live.index)]
y_win_live = y_live.reindex(X_win_live.index)
X_win = X_win_live.loc[X_win_live.index.intersection(X.index)]
y_win = y.reindex(X_win.index)
X_win_ts, y_win_ts = X_win.loc['FR'], y_win.loc['FR']

# Dernier pli de chaque découpage
train, test = list(cv_ts.split(X_win_ts))[-1]
X_win_ts_train, y_win_ts_train = X_win_ts.iloc[train], y_win_ts.iloc[train]
X_win_ts_test, y_win_ts_test = X_win_ts.iloc[test], y_win_ts.iloc[test]
train, test = list(cv_panel.split(X_win))[-1]
X_win_train, y_win_train = X_win.iloc[train], y_win.iloc[train]
X_win_test, y_win_test = X_win.iloc[test], y_win.iloc[test]

#### 3.3.2. Série temporelle

In [20]:
svr = TimeSeriesSVR(kernel='gak', C=10.0)
compare(y_win_ts_test, svr.fit(X_win_ts_train, y_win_ts_train).predict(X_win_ts_test))

MAE sur le test : 0.561


,observé,prévu
date,,
2024-05-01,4.013,3.785
2024-06-01,3.747,4.481
2024-07-01,3.887,4.708
2024-08-01,4.678,4.944
2024-09-01,4.683,3.841
2024-10-01,3.267,3.741


#### 3.3.3. Panel

Comme avec `sklearn`, un seul modèle est ajusté sur les fenêtres de toutes les entités.

In [21]:
knn = KNeighborsTimeSeriesRegressor(n_neighbors=5, metric='dtw')
compare(y_win_test, knn.fit(X_win_train, y_win_train).predict(X_win_test)).unstack('entity').head()

MAE sur le test : 1.250


observé                       prévu                     
entity          DE     ES     FR     IT     DE     ES     FR     IT
date                                                               
2024-05-01  -0.510  6.877  4.013  6.360 -0.312  4.518  1.782  5.382
2024-06-01  -2.606  4.770  3.747  6.204 -0.135  3.969  1.982  5.467
2024-07-01  -1.373  4.416  3.887  4.458  0.080  3.969  2.312  5.523
2024-08-01  -1.475  3.800  4.678  4.511  0.411  3.969  3.104  4.264
2024-09-01  -0.069  3.071  4.683  5.268 -2.006  3.375  2.756  5.212

#### 3.3.4. `Pipeline` et `XYPipeline`

In [22]:
pipeline = Pipeline([('scaler', StandardScaler()), ('model', TimeSeriesSVR(kernel='gak', C=10.0))])
pipeline.fit(X_win_train, y_win_train)
print(f'Pipeline, MAE : {mean_absolute_error(y_win_test, pipeline.predict(X_win_test)):.3f}')

pipeline = XYPipeline([('drop', DropNaNTarget()), ('model', knn)])
pipeline.fit(X_win_live, y_win_live)
X_win_now = X_win_live[X_win_live.index.get_level_values('date') == T]
pd.Series(pipeline.predict(X_win_now), index=X_win_now.index, name=f'prévision faite en {T:%Y-%m}').round(3)

Pipeline, MAE : 0.794


entity  date      
FR      2024-12-01    3.985
DE      2024-12-01   -0.791
IT      2024-12-01    2.820
ES      2024-12-01    1.215
Name: prévision faite en 2024-12, dtype: float64

#### 3.3.5. `cross_val_score`

In [23]:
for name, data, target, cv in [('série', X_win_ts, y_win_ts, cv_ts), ('panel', X_win, y_win, cv_panel)]:
    scores = cross_val_score(knn, data, target, cv=cv, scoring='neg_mean_absolute_error')
    print(f'{name} : MAE par pli {(-scores).round(3)}, moyenne {-scores.mean():.3f}')

série : MAE par pli [0.963 1.018 2.189 1.555], moyenne 1.431


panel : MAE par pli [0.837 1.345 1.579 1.25 ], moyenne 1.253


#### 3.3.6. `GridSearchCV`

In [24]:
search = GridSearchCV(
    knn,
    param_grid={'n_neighbors': [3, 5, 10], 'metric': ['dtw', 'euclidean']},
    cv=cv_val_panel,
    scoring='neg_mean_absolute_error',
).fit(X_win_train, y_win_train)

print(f'Meilleurs paramètres : {search.best_params_}, MAE de validation : {-search.best_score_:.3f}')
print(f'MAE sur le test      : {mean_absolute_error(y_win_test, search.predict(X_win_test)):.3f}')

Meilleurs paramètres : {'metric': 'euclidean', 'n_neighbors': 10}, MAE de validation : 1.093
MAE sur le test      : 1.042


#### 3.3.7. Limites et pièges

- Passer des covariables hétérogènes (tendance, mois, indicateur...) à un estimateur `tslearn` ne lève aucune erreur, mais compare des « séries » sans signification.
- `StandardScaler` standardise chaque retard séparément : la forme de la fenêtre change.
- Voisins et noyaux n'extrapolent pas au-delà des niveaux observés.
- Les distances élastiques coûtent cher : chaque paire de lignes comparées demande un temps quadratique en la longueur de la fenêtre.
- Les classes de `sktime.regression` et `sktime.classification` suivent la même logique de séries en entrée ; elles ne sont pas couvertes par un adapter de `tsforecast`.

### 3.4. Intégration des modèles `sktime`

#### 3.4.1. Principe

Un *forecaster* `sktime` raisonne en séries : `fit(y, X, fh)` puis `predict(fh, X)`, où `fh` est l'horizon de prévision. `SktimeAdapter(forecaster)` le transforme en régresseur `sklearn` :

| `sklearn` | `sktime` |
|---|---|
| `fit(X, y)` | `forecaster_.fit(y=y, X=X)` sur un clone du forecaster |
| `predict(X)` | `forecaster_.predict(fh, X=X)`, `fh` étant formé des **dates absolues** des lignes de `X` |
| `forecaster__<param>` | paramètre du forecaster |

L'adapter n'a pas de paramètre d'horizon : une ligne située `k` pas après la dernière ligne d'entraînement de son entité est une prévision à `k` pas **dans la série des origines**. L'horizon économique `h` est déjà porté par la construction de `y`.

Deux familles de forecasters se comportent différemment :

- `YfromX(regressor)` apprend `y` à partir de `X` ligne à ligne ;
- un forecaster autorégressif (`NaiveForecaster`...) ignore `X` et extrapole l'historique des cibles d'entraînement à travers le `gap`.

#### 3.4.2. Série temporelle

In [25]:
from sktime.forecasting.compose import YfromX
from sktime.forecasting.naive import NaiveForecaster

from tsforecast.adapters import SktimeAdapter

# Régression de y sur X
regression = SktimeAdapter(YfromX(Ridge())).fit(X_ts_train, y_ts_train)
compare(y_ts_test, regression.predict(X_ts_test))

MAE sur le test : 0.501


,observé,prévu
date,,
2024-05-01,4.013,3.180
2024-06-01,3.747,3.989
2024-07-01,3.887,4.328
2024-08-01,4.678,4.799
2024-09-01,4.683,5.154
2024-10-01,3.267,4.169


In [26]:
# Extrapolation de la dernière cible connue : X ignoré, même prévision sur toutes les lignes
naive = SktimeAdapter(NaiveForecaster(strategy='last')).fit(X_ts_train, y_ts_train)
compare(y_ts_test, naive.predict(X_ts_test))

MAE sur le test : 1.369


,observé,prévu
date,,
2024-05-01,4.013,2.677
2024-06-01,3.747,2.677
2024-07-01,3.887,2.677
2024-08-01,4.678,2.677
2024-09-01,4.683,2.677
2024-10-01,3.267,2.677


#### 3.4.3. Panel

Le panel est transmis au format natif de `sktime`. Un forecaster sans support natif du panel est ajusté **une fois par entité** : le modèle est local, contrairement à la régression mutualisée de la section 3.1.

In [27]:
regression = SktimeAdapter(YfromX(Ridge())).fit(X_train, y_train)
predictions = regression.predict(X_test)

# Contrôle : une régression de Ridge par entité donne les mêmes prévisions
per_entity = pd.concat([
    pd.Series(
        Ridge().fit(X_train.loc[[entity]], y_train.loc[[entity]]).predict(X_test.loc[[entity]]),
        index=X_test.loc[[entity]].index,
    )
    for entity in entities_small
]).reindex(X_test.index)
print(f'Écart maximal avec un Ridge par entité : {np.abs(predictions - per_entity).max():.1e}')

compare(y_test, predictions).unstack('entity').head()

Écart maximal avec un Ridge par entité : 0.0e+00


MAE sur le test : 0.778


observé                       prévu                     
entity          DE     ES     FR     IT     DE     ES     FR     IT
date                                                               
2024-05-01  -0.510  6.877  4.013  6.360 -1.446  5.485  3.180  5.293
2024-06-01  -2.606  4.770  3.747  6.204 -3.657  4.393  3.989  5.144
2024-07-01  -1.373  4.416  3.887  4.458 -3.238  3.820  4.328  4.386
2024-08-01  -1.475  3.800  4.678  4.511 -2.769  3.502  4.799  4.133
2024-09-01  -0.069  3.071  4.683  5.268 -1.053  3.120  5.154  4.928

Les lignes de `X` peuvent être dans n'importe quel ordre : les prévisions suivent cet ordre.

In [28]:
shuffled = X_test.sample(frac=1.0, random_state=0)
reordered = pd.Series(regression.predict(shuffled), index=shuffled.index).reindex(X_test.index)
print(f'Écart maximal après mélange des lignes : {np.abs(reordered - predictions).max():.1e}')

Écart maximal après mélange des lignes : 0.0e+00


#### 3.4.4. `Pipeline` et `XYPipeline`

L'adapter lit les dates et les entités dans l'index : les étapes qui le précèdent doivent rendre des `DataFrame` (`set_output(transform='pandas')`).

In [29]:
pipeline = Pipeline([('scaler', StandardScaler()), ('model', SktimeAdapter(YfromX(Ridge())))])
pipeline.set_output(transform='pandas')
pipeline.fit(X_train, y_train)
print(f'Pipeline, MAE : {mean_absolute_error(y_test, pipeline.predict(X_test)):.3f}')

pipeline = XYPipeline([('drop', DropNaNTarget()), ('model', SktimeAdapter(YfromX(Ridge())))])
pipeline.fit(X_live, y_live)
pd.Series(pipeline.predict(X_now), index=X_now.index, name=f'prévision faite en {T:%Y-%m}').round(3)

Pipeline, MAE : 0.786


entity  date      
FR      2024-12-01    5.107
DE      2024-12-01   -1.193
IT      2024-12-01    3.686
ES      2024-12-01    2.584
Name: prévision faite en 2024-12, dtype: float64

#### 3.4.5. `cross_val_score`

In [30]:
for label, forecaster in [('YfromX(Ridge)', YfromX(Ridge())), ('NaiveForecaster', NaiveForecaster())]:
    for name, data, target, cv in [('série', X_ts, y_ts, cv_ts), ('panel', X, y, cv_panel)]:
        scores = cross_val_score(
            SktimeAdapter(forecaster), data, target, cv=cv, scoring='neg_mean_absolute_error'
        )
        print(f'{label:<16} {name} : MAE par pli {(-scores).round(3)}, moyenne {-scores.mean():.3f}')

YfromX(Ridge)    série : MAE par pli [0.621 1.218 0.689 0.501], moyenne 0.757


YfromX(Ridge)    panel : MAE par pli [0.891 0.843 0.735 0.778], moyenne 0.812
NaiveForecaster  série : MAE par pli [1.11  1.483 0.91  1.369], moyenne 1.218


NaiveForecaster  panel : MAE par pli [2.522 2.292 2.145 1.221], moyenne 2.045


#### 3.4.6. `GridSearchCV`

La grille règle les paramètres du forecaster (`forecaster__<param>`), y compris ceux d'un estimateur imbriqué, et peut aussi comparer des forecasters entre eux.

In [31]:
search = GridSearchCV(
    SktimeAdapter(NaiveForecaster()),
    param_grid=[
        {'forecaster': [NaiveForecaster()], 'forecaster__strategy': ['last', 'mean', 'drift']},
        {'forecaster': [YfromX(Ridge())], 'forecaster__estimator__alpha': [0.1, 1.0, 10.0]},
    ],
    cv=cv_val_panel,
    scoring='neg_mean_absolute_error',
).fit(X_train, y_train)

print(f'Meilleurs paramètres : {search.best_params_}, MAE de validation : {-search.best_score_:.3f}')
print(f'MAE sur le test      : {mean_absolute_error(y_test, search.predict(X_test)):.3f}')

Meilleurs paramètres : {'forecaster': YfromX(estimator=Ridge()), 'forecaster__estimator__alpha': 0.1}, MAE de validation : 0.841
MAE sur le test      : 0.791


#### 3.4.7. Limites et pièges

- Les dates doivent appartenir à une grille régulière.
- Sur un panel, une entité inconnue à l'ajustement ne peut pas être prédite.
- Les forecasters qui exigent l'horizon dès l'ajustement (réductions directes) sont refusés.
- Un forecaster qui exige `X` à toutes les dates entre l'entraînement et la prévision (`SARIMAX`) échoue quand le `gap` est absent de `X` : aucune valeur n'est inventée.
- Un forecaster autorégressif extrapole la série des cibles décalées : sa prévision dépend de la distance à l'entraînement, et un forecaster qui a besoin de la fréquence (`NaiveForecaster`) demande au moins trois dates d'entraînement par entité sur un panel.

In [32]:
from sklearn.neighbors import KNeighborsRegressor
from sktime.forecasting.compose import make_reduction
from sktime.forecasting.sarimax import SARIMAX

# Entité inconnue
show_error(lambda: SktimeAdapter(NaiveForecaster())
           .fit(X_train.drop('ES', level='entity'), y_train.drop('ES', level='entity'))
           .predict(X_test))
# Horizon exigé à l'ajustement
show_error(lambda: SktimeAdapter(make_reduction(KNeighborsRegressor(), strategy='direct', window_length=3))
           .fit(X_ts_train, y_ts_train))
# Variables exogènes absentes sur le gap
show_error(lambda: SktimeAdapter(SARIMAX(order=(1, 0, 0)))
           .fit(X_ts_train, y_ts_train)
           .predict(X_ts_test))

ValueError : Entities unknown at fit: ('ES',)
ValueError : DirectTabularRegressionForecaster needs the forecasting horizon at fit; the horizon of SktimeAdapter is only known at predict (dates of the rows of X)


ValueError : SARIMAX needs X on every date between the end of training and the forecast date, but X has no row on '2024-03-01 00:00:00', '2024-04-01 00:00:00'; no value is invented for them: use a forecaster that accepts non-contiguous X (YfromX, ...) or no gap


### 3.5. Intégration des modèles `darts`

#### 3.5.1. Principe

Les modèles `darts` manipulent des objets `TimeSeries`. `DartsAdapter(model)` convertit les `DataFrame` et expose un régresseur `sklearn` ; c'est la **configuration du modèle** qui détermine son mode, lisible dans `mode_` :

| `mode_` | Modèle | Rôle de `X` | Lignes prévisibles |
|---|---|---|---|
| `'row'` | global sans retard de cible, `lags_future_covariates=[0]` | covariables de la date prévue | toutes, quel que soit le `gap`, y compris des entités inconnues |
| `'global'` | autre modèle global (retards de cible...) | covariables passées / futures selon les retards | après l'entraînement, entités connues |
| `'local'` | modèle local (`NaiveDrift`, `ExponentialSmoothing`, `ARIMA`...), un par entité | covariables futures si le modèle les accepte | après l'entraînement, entités connues |

Pour le paradigme « `y` à partir de `X` », utiliser le mode `'row'`. Les paramètres du modèle se règlent par `model__<param>`.

#### 3.5.2. Série temporelle

In [33]:
from darts.models import LinearRegressionModel, NaiveDrift, XGBModel

from tsforecast.adapters import DartsAdapter

# Mode ligne à ligne
row = DartsAdapter(LinearRegressionModel(lags_future_covariates=[0])).fit(X_ts_train, y_ts_train)
print(f'mode_ : {row.mode_}')
compare(y_ts_test, row.predict(X_ts_test))

mode_ : row
MAE sur le test : 0.514


,observé,prévu
date,,
2024-05-01,4.013,3.157
2024-06-01,3.747,3.987
2024-07-01,3.887,4.321
2024-08-01,4.678,4.801
2024-09-01,4.683,5.215
2024-10-01,3.267,4.167


In [34]:
# Contrôle : en mode ligne à ligne, le modèle équivaut à une régression linéaire sur X
reference = LinearRegression().fit(X_ts_train, y_ts_train).predict(X_ts_test)
print(f'Écart maximal avec LinearRegression : {np.abs(row.predict(X_ts_test) - reference).max():.1e}')

# Modes d'extrapolation : les cibles d'entraînement sont prolongées à travers le gap
for model in [LinearRegressionModel(lags=12), NaiveDrift()]:
    adapter = DartsAdapter(model).fit(X_ts_train, y_ts_train)
    mae = mean_absolute_error(y_ts_test, adapter.predict(X_ts_test))
    print(f'{type(model).__name__:<22} mode_ : {adapter.mode_:<7} MAE : {mae:.3f}')

Écart maximal avec LinearRegression : 9.8e-15
LinearRegressionModel  mode_ : global  MAE : 1.334
NaiveDrift             mode_ : local   MAE : 1.414


#### 3.5.3. Panel

Chaque entité forme sa propre `TimeSeries`. Un modèle global (modes `'row'` et `'global'`) est ajusté sur l'ensemble des séries, sans qu'une entité lise les covariables d'une autre.

In [35]:
boosting = DartsAdapter(XGBModel(lags_future_covariates=[0], n_estimators=100, max_depth=3, random_state=0))
boosting.fit(X_train, y_train)
print(f'mode_ : {boosting.mode_}')
compare(y_test, boosting.predict(X_test)).unstack('entity').head()

mode_ : row
MAE sur le test : 0.877


observé                       prévu                     
entity          DE     ES     FR     IT     DE     ES     FR     IT
date                                                               
2024-05-01  -0.510  6.877  4.013  6.360 -0.716  5.408  4.409  5.592
2024-06-01  -2.606  4.770  3.747  6.204 -0.745  4.655  5.050  5.619
2024-07-01  -1.373  4.416  3.887  4.458  0.010  3.736  5.168  4.278
2024-08-01  -1.475  3.800  4.678  4.511  0.572  3.703  5.584  4.176
2024-09-01  -0.069  3.071  4.683  5.268  0.594  3.748  5.428  4.867

In [36]:
# Mode ligne à ligne : une entité absente de l'entraînement reste prévisible
without_es = DartsAdapter(LinearRegressionModel(lags_future_covariates=[0]))
without_es.fit(X_train.drop('ES', level='entity'), y_train.drop('ES', level='entity'))
es_test = X_test.loc[['ES']]
print(f"MAE sur 'ES', jamais vue : {mean_absolute_error(y_test.loc[['ES']], without_es.predict(es_test)):.3f}")

MAE sur 'ES', jamais vue : 0.811


#### 3.5.4. `Pipeline` et `XYPipeline`

In [37]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DartsAdapter(LinearRegressionModel(lags_future_covariates=[0]))),
])
pipeline.set_output(transform='pandas')
pipeline.fit(X_train, y_train)
print(f'Pipeline, MAE : {mean_absolute_error(y_test, pipeline.predict(X_test)):.3f}')

pipeline = XYPipeline([('drop', DropNaNTarget()), ('model', boosting)])
pipeline.fit(X_live, y_live)
pd.Series(pipeline.predict(X_now), index=X_now.index, name=f'prévision faite en {T:%Y-%m}').round(3)

Pipeline, MAE : 0.639


entity  date      
FR      2024-12-01    5.368
DE      2024-12-01   -0.303
IT      2024-12-01    2.891
ES      2024-12-01    1.758
Name: prévision faite en 2024-12, dtype: float64

#### 3.5.5. `cross_val_score`

In [38]:
for model in [LinearRegressionModel(lags_future_covariates=[0]), NaiveDrift()]:
    for name, data, target, cv in [('série', X_ts, y_ts, cv_ts), ('panel', X, y, cv_panel)]:
        scores = cross_val_score(
            DartsAdapter(model), data, target, cv=cv, scoring='neg_mean_absolute_error'
        )
        print(f'{type(model).__name__:<22} {name} : MAE par pli {(-scores).round(3)}, moyenne {-scores.mean():.3f}')

LinearRegressionModel  série : MAE par pli [0.608 1.247 0.708 0.514], moyenne 0.769


LinearRegressionModel  panel : MAE par pli [0.769 1.367 1.156 0.639], moyenne 0.983
NaiveDrift             série : MAE par pli [1.168 1.492 0.969 1.414], moyenne 1.261
NaiveDrift             panel : MAE par pli [2.725 2.301 2.085 1.343], moyenne 2.113


#### 3.5.6. `GridSearchCV`

In [39]:
search = GridSearchCV(
    DartsAdapter(XGBModel(lags_future_covariates=[0], n_estimators=100, max_depth=3, random_state=0)),
    param_grid={'model__max_depth': [2, 3], 'model__n_estimators': [50, 100]},
    cv=cv_val_panel,
    scoring='neg_mean_absolute_error',
).fit(X_train, y_train)

print(f'Meilleurs paramètres : {search.best_params_}, MAE de validation : {-search.best_score_:.3f}')
print(f'MAE sur le test      : {mean_absolute_error(y_test, search.predict(X_test)):.3f}')

Meilleurs paramètres : {'model__max_depth': 3, 'model__n_estimators': 50}, MAE de validation : 1.119
MAE sur le test      : 0.889


#### 3.5.7. Limites et pièges

- Les dates doivent appartenir à une grille régulière.
- En modes `'global'` et `'local'`, les lignes d'entraînement de chaque entité doivent être contiguës, seules les dates postérieures à l'entraînement d'une entité connue sont prévisibles, et un modèle qui lit des covariables a besoin de `X` à toutes les dates intermédiaires, `gap` compris.
- En mode `'row'`, les encodeurs `position` sont refusés.
- Les classifieurs `darts` sont refusés ; les modèles `torch` passent par le mode `'global'` sans être testés avec l'adapter.
- Les covariables statiques ne sont pas gérées : ajouter des indicatrices d'entité dans `X`.

In [40]:
# Covariables absentes sur le gap
show_error(lambda: DartsAdapter(LinearRegressionModel(lags=3, lags_future_covariates=[0]))
           .fit(X_ts_train, y_ts_train)
           .predict(X_ts_test))
# Ligne antérieure à la fin de l'entraînement avec un modèle qui extrapole
show_error(lambda: DartsAdapter(NaiveDrift()).fit(X_ts_train, y_ts_train).predict(X_ts_train.tail(2)))
# Lignes d'entraînement trouées avec un modèle qui extrapole
holes = X_ts_train.index.month != 6
show_error(lambda: DartsAdapter(NaiveDrift()).fit(X_ts_train[holes], y_ts_train[holes]))

ValueError : LinearRegressionModel reads covariates on every date between the end of training and the forecast date, but X has no row on '2024-03-01 00:00:00', '2024-04-01 00:00:00'; no value is invented for them: use a model without covariates or no gap
ValueError : NaiveDrift extrapolates the target history: it cannot predict rows at or before the last training date of their entity ('2024-01-01 00:00:00', '2024-02-01 00:00:00')
ValueError : The training rows do not cover every date of the 'MS' grid; NaiveDrift extrapolates the target history and needs contiguous training rows


> ⚠️ **Section en cours de réécriture** (`hierarchicalforecast`) : le code ci-dessous est conservé en commentaire et n'est pas exécuté.

### 3.6. Intégration des modèles de `hierarchicalforecast`


Le package `hierarchicalforecast` est spécialisé dans la réconciliation de prévisions hiérarchiques. 
Lorsque des séries temporelles sont liées par des contraintes d'agrégation (géographiques, temporelles, 
catégorielles), les prévisions indépendantes de chaque série ne respectent pas nécessairement ces contraintes. 
La réconciliation ajuste les prévisions pour les rendre **cohérentes**.

Pour intégrer ces fonctionnalités dans un workflow unifié, `tsforecast` fournit l'adapter 
`HierarchicalForecastAdapter` qui encapsule les méthodes de réconciliation dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour construire la hiérarchie à partir des données historiques ;
- `.predict(X)` pour réconcilier des prévisions de base.

**Workflow typique :**
1. Entraîner des modèles indépendants pour chaque série de la hiérarchie
2. Générer des prévisions de base (potentiellement incohérentes)
3. Réconcilier ces prévisions avec l'adapter

Deux types de hiérarchies sont supportés :
- **Cross-sectionnelle** : agrégation entre séries (ex: Total = Composante_B + Composante_C)
- **Temporelle** : agrégation entre fréquences (ex: trimestriel = somme des mensuels)


#### 3.6.1. Réconciliation cross-sectionnelle

Dans cet exemple, on dispose de séries `value_b` et `value_c` dont la somme définit `value_a`. 
On entraîne des modèles indépendants sur chaque série, on génère des prévisions out-of-sample 
via `cross_val_predict`, puis on réconcilie ces prévisions pour restaurer la cohérence hiérarchique.


#### 3.6.1.1 Génération des prévisions indépendantes

In [41]:
# # Étape 1 : Préparation des features pour chaque composante
# # Construction de covariables simples (trend, mois, lags) pour chaque série
#
# # Fonction de construction des features pour la prédiction de chaque valeur
# def build_features(series: pd.Series, horizon: int) -> tuple[pd.DataFrame, pd.Series]:
#     """Build lag-based features for a univariate time series.
#
#     Args:
#         series: Time series with a DatetimeIndex or a MultiIndex whose last
#             level is a DatetimeIndex (panel format).
#         horizon: Forecast horizon (number of steps ahead).
#
#     Returns:
#         Tuple of (X, y) aligned for the given horizon, with NaN rows dropped.
#     """
#     # Initialisation du DataFrame de features
#     df_feat = pd.DataFrame(index=series.index)
#
#     # Création des features temporelles et des lags
#     df_feat["trend"] = np.arange(len(series))
#     df_feat["month"] = series.index.get_level_values(-1).month
#     for lag in [1, 2, 3, 6, 12]:
#         df_feat[f"lag_{lag}"] = series.shift(lag)
#
#     # Alignement de y[t+horizon] avec X[t]
#     y_aligned = series.shift(-horizon)
#
#     # Suppression des lignes incomplètes
#     mask = df_feat.notna().all(axis=1) & y_aligned.notna()
#     return df_feat.loc[mask], y_aligned.loc[mask]
#
# # Fonction d'agrégation par entité à une fréquence donnée
# def aggregate_panel(series: pd.Series, freq: str) -> pd.Series:
#     """Aggregate a monthly panel series to a lower temporal frequency.
#
#     Groups by entity and target period, summing monthly values within each
#     period.  The resulting index uses period-start timestamps so that
#     ``build_features`` and ``PanelOutOfSampleSplit`` remain compatible.
#
#     Args:
#         series: Monthly panel series with a MultiIndex (entity, date).
#         freq: Pandas offset alias for the target frequency, e.g. ``"QS"``
#             (quarter-start) or ``"YS"`` (year-start).
#
#     Returns:
#         Aggregated panel series with the same MultiIndex names (entity, date).
#     """
#     # Groupement par entité et période cible, puis sommation
#     return (
#         series
#         .groupby("entity")
#         .apply(func=lambda x : x.droplevel('entity').resample(freq).sum())
#     )
#
#
# # Horizon de prévision par fréquence
# horizon_m = 2   # mensuel  : 2 mois
# horizon_q = 1   # trimestriel : 1 trimestre
# horizon_y = 1   # annuel      : 1 an
#
# # Features mensuelles (hiérarchie cross-sectionnelle : a = b + c, c = d + e)
# X_b, y_b = build_features(series=df_hier["value_b"], horizon=horizon_m)
# X_d, y_d = build_features(series=df_hier["value_d"], horizon=horizon_m)
# X_e, y_e = build_features(series=df_hier["value_e"], horizon=horizon_m)
# X_c, y_c = build_features(series=df_hier["value_c"], horizon=horizon_m)
# X_a, y_a = build_features(series=df_hier["value_a"], horizon=horizon_m)
#
# # Agrégation temporelle du total vers fréquences trimestrielle et annuelle
# value_a_q = aggregate_panel(df_hier["value_a"], freq="QS")
# value_a_y = aggregate_panel(df_hier["value_a"], freq="YS")
#
# X_a_q, y_a_q = build_features(series=value_a_q, horizon=horizon_q)
# X_a_y, y_a_y = build_features(series=value_a_y, horizon=horizon_y)
#
# # Application du décalage d'horizon sur les features (double décalage intentionnel)
# X_a   = X_a.shift(-horizon_m)
# X_b   = X_b.shift(-horizon_m)
# X_c   = X_c.shift(-horizon_m)
# X_d   = X_d.shift(-horizon_m)
# X_e   = X_e.shift(-horizon_m)
# X_a_q = X_a_q.shift(-horizon_q)
# X_a_y = X_a_y.shift(-horizon_y)
#
# # Affichage
# print("Covariables mensuelles   :", X_b.shape, X_d.shape, X_e.shape, X_c.shape, X_a.shape)
# print("Covariables trimestrielles:", X_a_q.shape)
# print("Covariables annuelles     :", X_a_y.shape)

In [42]:
# # Étape 2 : Génération de prévisions out-of-sample via cross_val_predict
# # Importation du modèle de prévision
# from sklearn.linear_model import LinearRegression
# from sklearn.utils import _safe_indexing
#
# # Fonction d'itération d'une crossvall sur des listes de jeux de données
# def run_cv_loop(
#     cv_split,
#     Xs_train: list[pd.DataFrame],
#     ys_train: list[pd.Series],
#     Xs_test: list[pd.DataFrame],
#     full_index: pd.MultiIndex,
#     col_names: list[str],
# ) -> pd.DataFrame:
#     """Run a manual cross-validation loop and collect out-of-sample predictions.
#
#     Args:
#         cv_split: Cross-validator yielding (train_idx, test_idx) pairs.
#         Xs_train: List of feature matrices to train on (one per target).
#         ys_train: List of target series aligned with Xs_train.
#         Xs_test: List of feature matrices to predict from (one per target).
#         full_index: MultiIndex of the full dataset, used to reconstruct the
#             (entity, date) index of each test fold.
#         col_names: Column names for the returned DataFrame, one per target.
#
#     Returns:
#         DataFrame of predictions indexed by (entity, date).
#     """
#     model = LinearRegression()
#     records = []
#
#     for train_idx, test_idx in cv_split:
#         # Prédiction indépendante pour chaque cible
#         y_hats = []
#         for X_tr_full, y_tr_full, X_te_full in zip(Xs_train, ys_train, Xs_test):
#             X_tr = _safe_indexing(X_tr_full, train_idx)
#             y_tr = _safe_indexing(y_tr_full, train_idx)
#             X_te = _safe_indexing(X_te_full, test_idx)
#             y_hats.append(model.fit(X_tr, y_tr).predict(X_te))
#
#         # Reconstruction de l'index (entity, date) du fold de test
#         test_index = full_index[test_idx]
#         for (entity, date), *vals in zip(test_index, *y_hats):
#             records.append({"entity": entity, "date": date, **dict(zip(col_names, vals))})
#
#     return pd.DataFrame(records).set_index(["entity", "date"])
#
#
# # Fenêtres de test par fréquence
# test_dates_m = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", periods=24, freq="MS")]
# test_dates_q = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", periods=8,  freq="QS")]
# test_dates_y = [d.strftime("%Y-%m-%d") for d in pd.date_range("2021-01-01", periods=2,  freq="YS")]
#
# # Cross-validateurs pour chaque fréquence
# cv_m = PanelOutOfSampleSplit(test_indices=test_dates_m, test_size=1, gap=horizon_m)
# cv_q = PanelOutOfSampleSplit(test_indices=test_dates_q, test_size=1, gap=horizon_q)
# cv_y = PanelOutOfSampleSplit(test_indices=test_dates_y, test_size=1, gap=horizon_y)
#
# # Prévisions mensuelles (hiérarchie cross-sectionnelle)
# df_hat_m = run_cv_loop(
#     cv_split=cv_m.split(X_a, y_a),
#     Xs_train=[X_a, X_b, X_c, X_d, X_e],
#     ys_train=[y_a, y_b, y_c, y_d, y_e],
#     Xs_test=[X_a, X_b, X_c, X_d, X_e],
#     full_index=X_a.index,
#     col_names=["y_hat_a", "y_hat_b", "y_hat_c", "y_hat_d", "y_hat_e"],
# )
#
# # Prévisions trimestrielles (total agrégé)
# df_hat_q = run_cv_loop(
#     cv_split=cv_q.split(X_a_q, y_a_q),
#     Xs_train=[X_a_q],
#     ys_train=[y_a_q],
#     Xs_test=[X_a_q],
#     full_index=X_a_q.index,
#     col_names=["y_hat_a_q"],
# )
#
# # Prévisions annuelles (total agrégé)
# df_hat_y = run_cv_loop(
#     cv_split=cv_y.split(X_a_y, y_a_y),
#     Xs_train=[X_a_y],
#     ys_train=[y_a_y],
#     Xs_test=[X_a_y],
#     full_index=X_a_y.index,
#     col_names=["y_hat_a_y"],
# )
#
# # Diagnostics
# # Incohérence cross-sectionnelle (hiérarchie a = b + c, fréquence mensuelle)
# incoherence_cs_a = (df_hat_m["y_hat_a"] - (df_hat_m["y_hat_b"] + df_hat_m["y_hat_c"])).abs()
# # Incohérence cross-sectionnelle (hiérarchie a = b + c, fréquence mensuelle)
# incoherence_cs_c = (df_hat_m["y_hat_c"] - (df_hat_m["y_hat_d"] + df_hat_m["y_hat_e"])).abs()
#
# # Incohérences temporelles 
# # Agrégation des prévisions mensuelles et trimestrielles vers les fréquences supérieures
# y_hat_a_m2q = aggregate_panel(df_hat_m["y_hat_a"],       freq="QS")
# y_hat_a_m2y = aggregate_panel(df_hat_m["y_hat_a"],       freq="YS")
# y_hat_a_q2y = aggregate_panel(df_hat_q["y_hat_a_q"],     freq="YS")
#
# # Calcul des écarts après alignement sur l'index commun
# incoherence_m2q = (y_hat_a_m2q - df_hat_q["y_hat_a_q"]).abs().dropna()
# incoherence_m2y = (y_hat_a_m2y - df_hat_y["y_hat_a_y"]).abs().dropna()
# incoherence_q2y = (y_hat_a_q2y - df_hat_y["y_hat_a_y"]).abs().dropna()
#
# # Affichage
# print("=== Prévisions mensuelles (cross-section) ===")
# print(f"  Shape : {df_hat_m.shape}")
# print(f"  Incohérence moyenne |ŷ_a - (ŷ_b + ŷ_c)| = {incoherence_cs_a.mean():.4f}")
# print(f"  Incohérence max                          = {incoherence_cs_a.max():.4f}")
# print(f"  Incohérence moyenne |ŷ_c - (ŷ_d + ŷ_e)| = {incoherence_cs_c.mean():.4f}")
# print(f"  Incohérence max                          = {incoherence_cs_c.max():.4f}")
#
# print("\n=== Prévisions trimestrielles (agrégation temporelle) ===")
# print(f"  Shape : {df_hat_q.shape}")
# print("  Mensuel → Trimestriel :")
# print(f"    Incohérence moyenne = {incoherence_m2q.mean():.4f}")
# print(f"    Incohérence max     = {incoherence_m2q.max():.4f}")
#
# print("\n=== Prévisions annuelles (agrégation temporelle) ===")
# print(f"  Shape : {df_hat_y.shape}")
# print("  Mensuel → Annuel :")
# print(f"    Incohérence moyenne = {incoherence_m2y.mean():.4f}")
# print(f"    Incohérence max     = {incoherence_m2y.max():.4f}")
# print("  Trimestriel → Annuel :")
# print(f"    Incohérence moyenne = {incoherence_q2y.mean():.4f}")
# print(f"    Incohérence max     = {incoherence_q2y.max():.4f}")

#### 3.6.1.2 Réconciliation des prévisions

In [43]:
# # ========================================================================
# # Étape 3 : Réconciliation des prévisions
# # ========================================================================
#
# # Construction du jeu de données de prédictions à agréger
# Y_hat_cs = df_hat_m.copy()
# Y_hat_cs.columns = pd.Index(['A', 'B', 'C'], name="component")
# Y_hat_cs = Y_hat_cs.stack().swaplevel('component', 'date').rename("y")# .to_frame()
#
# Y_hat_cs.head()

In [44]:
# df_hier_long['y'].head()

## Test de la méthode de fit

In [45]:
# # Initialisation des arguments
# # Méthodes de réconciliation
# reconcilers=[BottomUp(), MinTrace(method='ols')]
# # Hierarchie à respecter
# hierarchy = {
#     'A' : ['B', 'C']
#     'C' : ['D', 'E']
# }
# # 
#

## Exemples dans le notebook

In [46]:
# # Importation de l'adapter et des méthodes de réconciliation
# from tsforecast.adapters import HierarchicalForecastAdapter
# from hierarchicalforecast.methods import BottomUp, MinTrace
#
# # Initialisation de l'adapter avec la spécification cross-sectionnelle.
# # La spécification inclut le niveau entité en préfixe de chaque niveau de la hiérarchie,
# # du niveau le plus agrégé (entity/category) au niveau bottom (entity/category/component).
# spec_cs = [['entity'], ['entity', 'component']]
#
# adapter_cs = HierarchicalForecastAdapter(
#     reconcilers=[BottomUp(), MinTrace(method='ols')],
#     spec=spec_cs,
# )
#
# # Entraînement sur les données historiques bottom-level.
# # L'adapter calcule automatiquement les agrégats intermédiaires via aggregate().
# adapter_cs.fit(None, df_hier_long['y'])
#
# # Informations sur la hiérarchie construite
# info = adapter_cs.get_hierarchy_info()
# print(f'Hiérarchie construite :')
# print(f'  Nombre total de séries : {info["n_series"]}')
# print(f'  Séries bottom-level : {info["n_bottom"]}')
# print(f'  Niveaux : {info["levels"]}')
# print(f'  Tags : {info["tags"]}')
#
# # Réconciliation à partir du DataFrame au format MultiIndex
# Y_reconciled = adapter_cs.predict(Y_hat)
#
# # Affichage
# print(f'\nPrévisions réconciliées : {Y_reconciled.shape}')
# print(f'Colonnes : {Y_reconciled.columns.tolist()}')
#
# # Vérification de la cohérence après réconciliation
# # Choix de l'entité
# entity = "DE"
# # Parcours des colonnes de méthodes (colonne contenant "/" = méthode de réconciliation)
# for method_col in [c for c in Y_reconciled.columns if '/' in c]:
#     # Extraction des valeurs pour les trois niveaux de la hiérarchie
#     total = Y_reconciled[Y_reconciled['unique_id'] == f'{entity}/Total'][method_col].values
#     b_vals = Y_reconciled[Y_reconciled['unique_id'] == f'{entity}/Total/B'][method_col].values
#     c_vals = Y_reconciled[Y_reconciled['unique_id'] == f'{entity}/Total/C'][method_col].values
#     max_err = np.max(np.abs(total - (b_vals + c_vals)))
#     # Affichage de l'erreur de cohérence résiduelle
#     print(f'  {method_col}: max |Total - (B+C)| = {max_err:.1e}')

#### 3.6.2. Réconciliation temporelle

La réconciliation temporelle agrège les prévisions à différentes fréquences. 
Par exemple, des prévisions mensuelles sont agrégées en prévisions trimestrielles. 
La réconciliation ajuste les prévisions pour que la somme des mensuels corresponde aux trimestriels.

L'adapter détecte automatiquement la colonne d'identifiant lorsque le `id_col` par défaut 
(`'unique_id'`) ne correspond pas au nom du niveau du MultiIndex.


In [47]:
# # Utilisation des données de panel générées précédemment (section 1)
# # y_train contient des données mensuelles avec MultiIndex (entity, date)
#
# # Spécification de la hiérarchie temporelle
# spec_temporal = {
#     'quarterly': 3,  # Agrégation trimestrielle (somme de 3 mois)
#     'monthly': 1,    # Niveau de base (mensuel)
# }
#
# # Initialisation de l'adapter
# # /!\ id_col='entity' correspond au nom du niveau du MultiIndex
# # Alternativement, l'adapter détecte automatiquement si 'unique_id' est absent
# adapter_temporal = HierarchicalForecastAdapter(
#     reconcilers=[BottomUp(), MinTrace(method='ols')],
#     spec=spec_temporal,
#     aggregation_type='local'
# )
#
# # Construction de la hiérarchie temporelle
# adapter_temporal.fit(None, y_train)
#
# # Informations sur la hiérarchie
# info_temp = adapter_temporal.get_hierarchy_info()
# print(f'Hiérarchie temporelle construite :')
# print(f'  Nombre total de séries : {info_temp["n_series"]}')
# print(f'  Séries bottom-level : {info_temp["n_bottom"]}')
# print(f'  Niveaux : {info_temp["levels"]}')
#
# # Aperçu des données agrégées
# print(f'\nAperçu Y_df_ (données agrégées) :')
# print(adapter_temporal.Y_df_.head(10))

#### 3.6.3. Évaluation avec `cross_val_score`

L'adapter est compatible avec `cross_val_score` de `sklearn`. Dans ce contexte :
- `X` contient les **prévisions de base** à réconcilier
- `y` contient les **valeurs réelles** pour évaluer la qualité
- `fit(X_train, y_train)` construit la hiérarchie à partir de `y_train`
- `score(X_test, y_test)` réconcilie `X_test` et compare à `y_test`

Pour cet exemple simplifié, on utilise les valeurs réelles comme prévisions de base 
(i.e. les "prévisions" sont déjà parfaites avant réconciliation).


In [48]:
# # Importation de cross_val_score
# from sklearn.model_selection import cross_val_score
#
# # Définition de l'adapter avec hiérarchie temporelle
# adapter_cv = HierarchicalForecastAdapter(
#     reconcilers=[BottomUp()],
#     spec={'quarterly': 3, 'monthly': 1},
# )
#
# # Évaluation avec cross-validation
# # X=y : les "prévisions de base" sont les vraies valeurs (cas simplifié)
# scores_hierarchical = cross_val_score(
#     estimator=adapter_cv,
#     X=y,
#     y=y,
#     cv=cv,
#     scoring='r2'
# )
#
# print(f'Scores R² par fold : {scores_hierarchical}')
# print(f'Score R² moyen : {scores_hierarchical.mean():.4f} (+/- {scores_hierarchical.std():.4f})')
#

#### 3.6.4. Intégration dans une `Pipeline` sklearn

L'adapter peut être intégré comme étape finale d'une `Pipeline` sklearn. 
Les étapes de preprocessing transforment les prévisions de base avant réconciliation.

**Attention :** Dans cette configuration, `X` représente les prévisions de base (et non des covariables) 
et les transformers s'appliquent à ces prévisions. Le `fit` de l'adapter utilise `y` pour construire la hiérarchie.


In [49]:
# # Importation de la pipeline et du scaler
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
#
# # Construction d'une pipeline : preprocessing + réconciliation
# pipeline_hier = Pipeline([
#     ('scaler', StandardScaler()),
#     ('reconciler', HierarchicalForecastAdapter(
#         reconcilers=[BottomUp()],
#         spec={'quarterly': 3, 'monthly': 1},
#     ))
# ])
#
# # Propagation du format pandas
# pipeline_hier.set_output(transform='pandas')
#
# # Entraînement de la pipeline
# # /!\ X=y car les prévisions de base sont ici les valeurs réelles
# pipeline_hier.fit(X=y_train, y=y_train)
#
# # Prédiction (réconciliation des prévisions transformées)
# y_pred_pipeline_hier = pipeline_hier.predict(X_test)
#
# # Affichage
# print(f'Prédictions via Pipeline : {type(y_pred_pipeline_hier)}')
# if hasattr(y_pred_pipeline_hier, 'shape'):
#     print(f'Shape : {y_pred_pipeline_hier.shape}')
# if hasattr(y_pred_pipeline_hier, 'head'):
#     print(y_pred_pipeline_hier.head())

#### 3.6.5. Optimisation avec `GridSearchCV`

`GridSearchCV` peut être utilisé pour comparer différentes configurations de réconciliation :
- Type d'agrégation (`local` vs `global`)
- Spécification temporelle (différents niveaux d'agrégation)

**Note :** Les méthodes de réconciliation (`reconcilers`) étant des objets instanciés, 
leur exploration nécessite de passer des listes complètes comme valeurs de la grille.


In [50]:
# # Importation de GridSearchCV
# from sklearn.model_selection import GridSearchCV
#
# # Définition de la grille de paramètres
# # Comparaison de différentes méthodes de réconciliation et types d'agrégation
# param_grid = {
#     'reconcilers': [
#         [BottomUp()],
#         [MinTrace(method='ols')],
#         [MinTrace(method='mint_shrink')],
#     ],
#     'aggregation_type': ['local'],
# }
#
# # Initialisation de l'estimateur de base
# adapter_grid = HierarchicalForecastAdapter(
#     reconcilers=[BottomUp()],
#     spec={'quarterly': 3, 'monthly': 1},
#     id_col='entity',
# )
#
# # Initialisation du GridSearchCV
# grid_search_hier = GridSearchCV(
#     estimator=adapter_grid,
#     param_grid=param_grid,
#     cv=cv_val,
#     n_jobs=1,  # Sérialisation pour éviter les conflits de mémoire
#     scoring='r2',
#     error_score='raise',
# )
#
# # Entraînement avec recherche d'hyperparamètres
# # X=y car les prévisions de base sont ici les valeurs réelles
# grid_search_hier.fit(X=y, y=y)
#
# # Résultats
# print(f'Meilleurs paramètres : {grid_search_hier.best_params_}')
# print(f'Meilleur score R² : {grid_search_hier.best_score_:.4f}')
#
# # Détails des résultats
# results = pd.DataFrame(grid_search_hier.cv_results_)
# print(f'\nRésumé des résultats :')
# print(results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']])
#

> ⚠️ **Section en cours de réécriture** (`opera`) : le code ci-dessous est conservé en commentaire et n'est pas exécuté.

### 3.7. Intégration des modèles de `opera`

Le package `opera` se spécialise dans l'agrégation en ligne d'experts (online expert aggregation), permettant de combiner dynamiquement les prévisions de plusieurs modèles. Son API spécifique nécessite des formats particuliers.

Pour intégrer ces fonctionnalités dans un workflow unifié, `tsforecast` fournit l'adapter `OperaAdapter` qui encapsule les algorithmes d'agrégation dans une interface compatible `sklearn`.

La syntaxe devient alors celle de `sklearn` :
- `.fit(X, y)` pour entraîner le système d'agrégation sur les prédictions des experts ;
- `.predict(X)` pour combiner les prédictions des experts ;
- `.partial_fit(X, y)` pour mise à jour incrémentale (online learning) ;
- `.score(X, y)` pour évaluer la qualité de l'ensemble ;

L'adapter gère automatiquement :
- La conversion des formats de données
- L'initialisation des coefficients d'experts
- La mise à jour des poids selon les performances passées
- Le calcul des prédictions d'ensemble

**Cas d'usage typique:** Vous avez entraîné plusieurs modèles différents (ex: Ridge, Lasso, RandomForest) et vous voulez les combiner de manière optimale en fonction de leurs performances.

In [51]:
# # Importation de l'adapter
# from tsforecast.adapters import OperaAdapter
# # Importation de modèles sklearn pour créer des "experts"
# from sklearn.linear_model import Ridge, Lasso
# from sklearn.ensemble import RandomForestRegressor
#
# # Étape 1: Entraîner plusieurs modèles "experts"
# print("Entraînement des modèles experts...")
# expert_ridge = Ridge(alpha=1.0).fit(X_train, y_train)
# expert_lasso = Lasso(alpha=1.0).fit(X_train, y_train)
# expert_rf = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train, y_train)
#
# # Étape 2: Obtenir leurs prédictions (qui serviront d'input à opera)
# expert_predictions_train = np.column_stack([
#     expert_ridge.predict(X_train),
#     expert_lasso.predict(X_train),
#     expert_rf.predict(X_train)
# ])
#
# expert_predictions_test = np.column_stack([
#     expert_ridge.predict(X_test),
#     expert_lasso.predict(X_test),
#     expert_rf.predict(X_test)
# ])
#
# print(f"Prédictions des experts (train): {expert_predictions_train.shape}")
# print(f"Prédictions des experts (test): {expert_predictions_test.shape}")
#
# # Étape 3: Entraîner l'adapter Opera pour combiner les experts
# estimator_opera = OperaAdapter(
#     model="BOA",  # Bernstein Online Aggregation
#     loss_type="mse"
# )
#
# # Entraînement sur les prédictions des experts
# estimator_opera.fit(expert_predictions_train, y_train.values)
#
# # Étape 4: Combiner les prédictions des experts sur le test
# y_pred_opera = estimator_opera.predict(expert_predictions_test)
#
# # Évaluation
# r2_opera = estimator_opera.score(expert_predictions_test, _safe_indexing(y, test).values)
# print(f"\nScore R² de l'ensemble: {r2_opera:.4f}")
#
# # Comparaison avec les experts individuels
# from sklearn.metrics import r2_score
# print(f"Score R² Ridge: {r2_score(_safe_indexing(y, test), expert_ridge.predict(X_test)):.4f}")
# print(f"Score R² Lasso: {r2_score(_safe_indexing(y, test), expert_lasso.predict(X_test)):.4f}")
# print(f"Score R² RandomForest: {r2_score(_safe_indexing(y, test), expert_rf.predict(X_test)):.4f}")
#
# y_pred_opera

`GridSearchCV` peut être utilisé pour optimiser le choix de l'algorithme d'agrégation et du type de perte.

In [52]:
# # Importation de GridSearchCV
# from sklearn.model_selection import GridSearchCV
# from tsforecast.adapters import OperaAdapter
#
# # Préparation des prédictions des experts sur l'ensemble complet
# expert_predictions_full = np.column_stack([
#     expert_ridge.predict(X),
#     expert_lasso.predict(X),
#     expert_rf.predict(X)
# ])
#
# # Définition de la grille de paramètres
# param_grid = {
#     'model': ['BOA', 'EWA', 'MLpol'],  # Différents algorithmes d'agrégation
#     'loss_type': ['mse', 'mae'],  # Différentes fonctions de perte
# }
#
# # Initialisation du GridSearchCV
# grid_search_opera = GridSearchCV(
#     estimator=OperaAdapter(),
#     param_grid=param_grid,
#     cv=cv,
#     n_jobs=-1,
#     scoring='neg_mean_squared_error'
# )
#
# # Entraînement
# print("Recherche des meilleurs hyperparamètres...")
# grid_search_opera.fit(expert_predictions_full, y.values)
#
# # Meilleurs paramètres et score
# print(f"\nMeilleurs paramètres: {grid_search_opera.best_params_}")
# print(f"Meilleur score: {grid_search_opera.best_score_:.4f}")
#
# # Prédiction avec le meilleur modèle
# y_pred_grid_opera = grid_search_opera.predict(expert_predictions_test)
#
# # Comparaison des performances
# r2_grid = r2_score(_safe_indexing(y, test), y_pred_grid_opera)
# print(f"Score R² (meilleur modèle): {r2_grid:.4f}")
#
# y_pred_grid_opera

Une particularité d'`OperaAdapter` est le support de l'apprentissage incrémental via `partial_fit`, permettant de mettre à jour les poids des experts au fur et à mesure que de nouvelles données arrivent (online learning).

In [53]:
# # Simulation d'un scénario d'apprentissage incrémental
# # Divisons les données d'entraînement en batches
#
# # Initialisation de l'adapter
# estimator_opera_online = OperaAdapter(model="EWA", loss_type="mse")  # Exponentially Weighted Average
#
# # Premier batch: entraînement initial
# batch_size = len(X_train) // 3
# expert_preds_batch1 = expert_predictions_train[:batch_size]
# y_batch1 = y_train.values[:batch_size]
#
# print(f"Entraînement initial sur {batch_size} observations...")
# estimator_opera_online.fit(expert_preds_batch1, y_batch1)
#
# # Batches suivants: mise à jour incrémentale
# for i in range(1, 3):
#     start_idx = i * batch_size
#     end_idx = (i + 1) * batch_size if i < 2 else len(X_train)
#
#     expert_preds_batch = expert_predictions_train[start_idx:end_idx]
#     y_batch = y_train.values[start_idx:end_idx]
#
#     print(f"Mise à jour incrémentale {i}: {len(y_batch)} observations...")
#     estimator_opera_online.partial_fit(expert_preds_batch, y_batch)
#
# # Prédiction finale
# y_pred_opera_online = estimator_opera_online.predict(expert_predictions_test)
#
# # Évaluation
# r2_online = estimator_opera_online.score(expert_predictions_test, _safe_indexing(y, test).values)
# print(f"\nScore R² (apprentissage incrémental): {r2_online:.4f}")
#
# y_pred_opera_online